# Rossmann Sales EDA — Notebook nâng cao

Notebook này tổng hợp phân tích dữ liệu gốc, thống kê doanh thu, dữ liệu thiếu và biểu đồ trực quan để hỗ trợ đánh giá mô hình.

## Mục tiêu
- Khám phá xu hướng doanh thu theo thời gian
- Đánh giá tác động khuyến mãi và ngày trong tuần
- Kiểm tra dữ liệu thiếu trong store.csv
- Chuẩn bị nền tảng cho mô hình dự đoán


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

root = Path.cwd().resolve().parents[1]
train = pd.read_csv(root / 'data/raw/train.csv', parse_dates=['Date'], low_memory=False)
test = pd.read_csv(root / 'data/raw/test.csv', parse_dates=['Date'], low_memory=False)
store = pd.read_csv(root / 'data/raw/store.csv', low_memory=False)
print('train shape:', train.shape)
print('test shape:', test.shape)
print('store shape:', store.shape)
train.head()


In [ ]:
monthly = (
    train.assign(Month=train.Date.dt.to_period('M').astype(str))
    .groupby('Month')
    .agg(Sales=('Sales', 'sum'))
    .reset_index()
)
monthly['Sales_M'] = monthly['Sales'] / 1_000_000
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(monthly['Month'], monthly['Sales_M'], color='#4C78A8')
ax.set_title('Doanh thu theo tháng (triệu EUR)')
ax.set_xlabel('Tháng')
ax.set_ylabel('Triệu EUR')
plt.xticks(rotation=45)
plt.tight_layout()
fig


In [ ]:
weekday = (
    train.assign(Weekday=train.Date.dt.day_name())
    .groupby('Weekday')['Sales']
    .mean()
    .reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(weekday.index, weekday.values, color='#F58518')
ax.set_title('Doanh thu trung bình theo ngày trong tuần')
ax.set_xlabel('Ngày trong tuần')
ax.set_ylabel('EUR')
plt.xticks(rotation=20)
plt.tight_layout()
fig


In [ ]:
promo = train.groupby('Promo')['Sales'].agg(['mean', 'median', 'count']).reset_index()
promo['PromoLabel'] = promo['Promo'].map({0: 'Không promo', 1: 'Có promo'})
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(promo['PromoLabel'], promo['mean'], color=['#7AA6D8', '#E45756'])
ax.set_title('Tác động khuyến mãi lên doanh thu')
ax.set_xlabel('Promo')
ax.set_ylabel('Doanh thu trung bình (EUR)')
plt.tight_layout()
fig


In [ ]:
open_effect = train.groupby('Open')['Sales'].agg(['mean', 'median', 'count']).reset_index()
open_effect['OpenLabel'] = open_effect['Open'].map({0: 'Đóng cửa', 1: 'Mở cửa'})
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(open_effect['OpenLabel'], open_effect['mean'], color=['#A0AEC0', '#54A24B'])
ax.set_title('Doanh thu trung bình theo trạng thái mở cửa')
ax.set_xlabel('Open')
ax.set_ylabel('Doanh thu trung bình (EUR)')
plt.tight_layout()
fig


In [ ]:
holiday = (
    train.assign(StateHolidayLabel=train['StateHoliday'].astype(str).replace({
        '0': 'Không nghỉ', 'a': 'Ngày lễ công cộng',
        'b': 'Lễ Phục sinh', 'c': 'Giáng sinh'
    }))
    .groupby('StateHolidayLabel')['Sales'].agg(['mean', 'median', 'count']).reset_index()
)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(holiday['StateHolidayLabel'], holiday['mean'], color='#9467BD')
ax.set_title('Doanh thu trung bình theo StateHoliday')
ax.set_xlabel('Loại ngày lễ')
ax.set_ylabel('Doanh thu trung bình (EUR)')
plt.xticks(rotation=20)
plt.tight_layout()
fig


In [ ]:
missing = store.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(missing.index.astype(str), missing.values, color='#B279A2')
ax.set_title('Dữ liệu thiếu trong store.csv')
ax.set_xlabel('Trường dữ liệu')
ax.set_ylabel('Số ô thiếu')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2, str(int(bar.get_height())), ha='center', va='bottom')
plt.xticks(rotation=45)
plt.tight_layout()
fig


## Kết luận
- Cuối tuần và ngày có promo đều có tác động rõ ràng lên doanh thu.
- Cần kiểm soát dữ liệu thiếu trong `CompetitionOpenSinceMonth`, `Promo2SinceWeek`, và `PromoInterval`.
- Mô hình nên dựa trên chuỗi thời gian và không xáo trộn ngẫu nhiên để tránh rò rỉ dữ liệu.
